# Step1：收集整理知识库

PDF_PATH 可以换成自己的 pdf 文件

In [1]:
# 导入所需库
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_deepseek import ChatDeepSeek
from langchain.chains import RetrievalQA
import os
from dotenv import load_dotenv

load_dotenv()

# ========== 知识库配置 ==========
PDF_PATH = "低空无人机集群通信.pdf"

# 检查文件是否存在
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"找不到 PDF 文件：{PDF_PATH}，请检查路径是否正确")

print(f"✅ 知识库文件：{PDF_PATH}")
print(f"   文件大小：{os.path.getsize(PDF_PATH) / 1024:.1f} KB")

/Users/a211026/Desktop/ai/RAG-demo/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/a211026/Desktop/ai/RAG-demo/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 知识库文件：低空无人机集群通信.pdf
   文件大小：843.3 KB


# Step2：从 PDF 中提取文本并记录页码

In [2]:
# PyMuPDFLoader 会自动为每页文本记录页码元数据（metadata.page）
loader = PyMuPDFLoader(PDF_PATH)
documents = loader.load()

print(f"📄 PDF 共提取了 {len(documents)} 页文本\n")

# 展示每页的文本长度和页码，验证页码元数据是否正确记录
print("页码 | 字符数 | 内容预览")
print("-" * 60)
for doc in documents:
    page_num = doc.metadata.get("page", "未知") + 1  # PyMuPDF 页码从0开始，+1转为实际页码
    content_preview = doc.page_content[:50].replace("\n", " ")
    print(f"第{page_num:>3}页 | {len(doc.page_content):>5}字 | {content_preview}...")

📄 PDF 共提取了 13 页文本

页码 | 字符数 | 内容预览
------------------------------------------------------------
第  1页 |  2975字 | http://www.paper.edu.cn  - 1 -  中国科技论文在线         融...
第  2页 |  1206字 | http://www.paper.edu.cn  - 2 -  中国科技论文在线 0 引言  随着信...
第  3页 |   932字 | http://www.paper.edu.cn  - 3 -  中国科技论文在线 在网络拓扑不断变化...
第  4页 |  1027字 | http://www.paper.edu.cn  - 4 -  中国科技论文在线 ∑Bi N i=1...
第  5页 |  1095字 | http://www.paper.edu.cn  - 5 -  中国科技论文在线 2.2  Q-le...
第  6页 |  1085字 | http://www.paper.edu.cn  - 6 -  中国科技论文在线 3.1  算法设计...
第  7页 |   328字 | http://www.paper.edu.cn  - 7 -  中国科技论文在线 该奖励函数能够鼓励...
第  8页 |    49字 | http://www.paper.edu.cn  - 8 -  中国科技论文在线 3.6 算法流程...
第  9页 |   837字 | http://www.paper.edu.cn  - 9 -  中国科技论文在线 4 基于Q-lea...
第 10页 |   759字 | http://www.paper.edu.cn  - 10 -  中国科技论文在线 OLSR 是一种...
第 11页 |   629字 | http://www.paper.edu.cn  - 11 -  中国科技论文在线 图4-1 给出了...
第 12页 |   961字 | http://www.paper.edu.cn  - 12 -  中国科技论文在线 图4-3 给出了...
第 13页 |   987字 | http://www.paper.edu.cn  - 13 -  中国科

# Step3：处理文本并创建向量存储

将文本分成小段并向量化并存入 FAISS

In [3]:
# --- 3.1 文本分块 ---
# 使用 RecursiveCharacterTextSplitter 按语义边界递归切分
# 切分后每个 chunk 会自动继承原始文档的 metadata（包含页码信息）
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "；", "，", " "]
)
docs = text_splitter.split_documents(documents)

print(f"📦 文档已切分成 {len(docs)} 个文本块")
print(f"\n示例 — 第1个块：")
print(f"  来源页码：第{docs[0].metadata.get('page', 0) + 1}页")
print(f"  文本内容：{docs[0].page_content[:100]}...")

📦 文档已切分成 35 个文本块

示例 — 第1个块：
  来源页码：第1页
  文本内容：http://www.paper.edu.cn 
- 1 - 
中国科技论文在线
      
 融合Q-learning 的低空无人机集群通信链
路自适应调度研究  
杨锦涛，王昕** 
（辽宁工程...


In [4]:
# --- 3.2 向量化并存入 FAISS ---
# 使用本地免费的中文向量模型，无需 API Key
embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")

# 构建向量库并持久化到本地
db = FAISS.from_documents(docs, embeddings)
db.save_local("faiss_index_pdf")

print(f"✅ 向量库构建完成，共索引 {len(docs)} 个文本块，已保存到 faiss_index_pdf/")

/var/folders/3t/8_rvw5nj0gs2j9kwyv8n0mhr0000gp/T/ipykernel_1782/2694171849.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")


✅ 向量库构建完成，共索引 35 个文本块，已保存到 faiss_index_pdf/


# Step4：执行相似度搜索

In [5]:
# 在调用 LLM 之前，先单独执行相似度搜索，查看检索到了哪些文档块
# 这一步不消耗 API Token，只使用本地向量模型
search_query = "低空无人机集群通信面临哪些主要挑战？"

# similarity_search_with_score 返回文档和相似度分数（分数越小越相似）
search_results = db.similarity_search_with_score(search_query, k=3)

print(f"🔍 查询：{search_query}")
print(f"📋 检索到 {len(search_results)} 个相关文档块：\n")

for i, (doc, score) in enumerate(search_results, 1):
    page_num = doc.metadata.get("page", 0) + 1
    print(f"--- 结果 {i} | 相似度分数：{score:.4f} | 来源：第{page_num}页 ---")
    print(f"{doc.page_content[:200]}...\n")

🔍 查询：低空无人机集群通信面临哪些主要挑战？
📋 检索到 3 个相关文档块：

--- 结果 1 | 相似度分数：169.8385 | 来源：第2页 ---
http://www.paper.edu.cn 
- 2 - 
中国科技论文在线
0 引言 
随着信息技术与航空技术的发展，无人机在多领域广泛应用，低空经济推动下的无人机
集群协同作业催生了低空无人机自组网（FANET），其可在通信基础设施薄弱或受损环境
50 
中实现灵活高效信息传输。与传统地面移动自组网相比，FANET 存在节点移动快、拓扑多
变、链路不稳定及节点资源受限等特点，传统路由协议难...

--- 结果 2 | 相似度分数：222.2666 | 来源：第13页 ---
升算法在大规模网络中的学习能力和泛化性能，并结合实际应用场景对算法进行优化与验
315 
证，从而进一步提高低空无人机自组网的实用性和可靠性。 
 
[参考文献] (References) 
[1] 祝源. 基于机器学习的无人机组网技术研究[D]. 成都：电子科技大学，2024. 
[2] 梁五建. 基于消息摆渡的无人集群网络路由技术研究[D]. 北京：北京邮电大学，2024. 
320 
[3]...

--- 结果 3 | 相似度分数：235.5969 | 来源：第3页 ---
http://www.paper.edu.cn 
- 3 - 
中国科技论文在线
在网络拓扑不断变化的情况下，邻居节点集合随时间动态更新，因此路由算法需要具备
良好的自适应能力，以保证通信的连续性和稳定性[2]。 
80 
1.2 
通信模型 
在低空无人机自组网中，节点之间通过无线信道进行数据传输，其通信性能受节点间距
离、信道衰减以及网络负载等因素影响。由于无人机网络具有较强的动态性，通信链路...



# Step5：使用问答链回答用户问题

In [6]:
# 从 .env 文件中获取 API 密钥
api_key = os.getenv("DEEPSEEK_API_KEY")
if api_key is None:
    raise ValueError("请在 .env 文件中设置 DEEPSEEK_API_KEY")

os.environ["DEEPSEEK_API_KEY"] = api_key

# 初始化 DeepSeek 模型
llm = ChatDeepSeek(model="deepseek-chat")

# 创建问答链，return_source_documents=True 让链返回引用的源文档
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

print("✅ 问答链创建完成！")

✅ 问答链创建完成！


In [7]:
# 提问并获取答案
query = "低空无人机集群通信面临哪些主要挑战？"

try:
    result = qa.invoke(query)
    print("🔍 问题:", query)
    print("🤖 答案:", result["result"])
except Exception as e:
    msg = str(e)
    if "402" in msg and "Insufficient Balance" in msg:
        print("⚠️ DeepSeek API 余额不足（402）。请充值或更换有余额的 API Key。")
    else:
        print("提问失败：", msg)

🔍 问题: 低空无人机集群通信面临哪些主要挑战？
🤖 答案: 根据提供的资料，低空无人机集群通信面临的主要挑战包括：

1. **节点移动速度快**：导致网络拓扑结构频繁且剧烈变化。
2. **拓扑多变**：网络结构不稳定，难以维持固定的通信路径。
3. **链路不稳定**：通信链路质量随时间变化，受节点间距离、信道衰减、网络负载等因素影响。
4. **节点资源受限**：无人机的计算、能量、存储等资源有限，限制了协议复杂度和通信能力。

这些特点使得传统路由协议（如AODV、OLSR）难以适配，容易导致路由失效、数据包丢失等问题。


# Step6：显示每个文档块的来源页码

In [8]:
# 展示 LLM 回答时引用的源文档块及其页码
if "source_documents" in result:
    source_docs = result["source_documents"]
    print(f"📚 本次回答引用了 {len(source_docs)} 个文档块：\n")
    
    for i, doc in enumerate(source_docs, 1):
        page_num = doc.metadata.get("page", 0) + 1
        source_file = doc.metadata.get("source", "未知")
        print(f"{'='*50}")
        print(f"📖 文档块 {i}")
        print(f"   来源文件：{source_file}")
        print(f"   来源页码：第 {page_num} 页")
        print(f"   文本内容：")
        print(f"   {doc.page_content[:300]}")
        if len(doc.page_content) > 300:
            print(f"   ...（共{len(doc.page_content)}字）")
        print()
else:
    print("⚠️ 未返回源文档信息，请确认问答链设置了 return_source_documents=True")

📚 本次回答引用了 3 个文档块：

📖 文档块 1
   来源文件：低空无人机集群通信.pdf
   来源页码：第 2 页
   文本内容：
   http://www.paper.edu.cn 
- 2 - 
中国科技论文在线
0 引言 
随着信息技术与航空技术的发展，无人机在多领域广泛应用，低空经济推动下的无人机
集群协同作业催生了低空无人机自组网（FANET），其可在通信基础设施薄弱或受损环境
50 
中实现灵活高效信息传输。与传统地面移动自组网相比，FANET 存在节点移动快、拓扑多
变、链路不稳定及节点资源受限等特点，传统路由协议难以适配，导致路由失效、数据包丢
失等问题，因此研究适配的智能路由与资源调度方法至关重要。国内外学者围绕无人机自组
网路由与资源管理开展大量研究，传统AODV、OLSR 协议因适配性不足难以满足需求，后
   ...（共484字）

📖 文档块 2
   来源文件：低空无人机集群通信.pdf
   来源页码：第 13 页
   文本内容：
   升算法在大规模网络中的学习能力和泛化性能，并结合实际应用场景对算法进行优化与验
315 
证，从而进一步提高低空无人机自组网的实用性和可靠性。 
 
[参考文献] (References) 
[1] 祝源. 基于机器学习的无人机组网技术研究[D]. 成都：电子科技大学，2024. 
[2] 梁五建. 基于消息摆渡的无人集群网络路由技术研究[D]. 北京：北京邮电大学，2024. 
320 
[3] 刘文睿. 面向无人机网络的感知通信融合路由技术研究[D]. 北京：北京邮电大学，2023. 
[4] 张昕婷. 面向电磁频谱侦察的无人机集群分簇机制与装订路由算法研究[D]. 南京：南京航空航天大学
   ...（共492字）

📖 文档块 3
   来源文件：低空无人机集群通信.pdf
   来源页码：第 3 页
   文本内容：
   http://www.paper.edu.cn 
- 3 - 
中国科技论文在线
在网络拓扑不断变化的情况下，邻居节点集合随时间动态更新，因此路由算法需要具备
良好的自适应能力，以保证通信的连续性和稳定性[2]。 
80 
1.2 
通信模型 
在低空无人机自组网中，节点之间通过无线信道进行数据传输，其通信性能受节点间距
离、信道衰减以及网络负载等因素影响。